# Sales Prediction using Advertising Data

This notebook performs exploratory data analysis for the Kaggle Advertising.csv dataset. The target variable is `Sales`, and the predictor variables are `TV`, `Radio`, and `Newspaper` advertising budgets.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "Advertising.csv"
FEATURES = ["TV", "Radio", "Newspaper"]
TARGET = "Sales"

## 1. Dataset Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed|^$", regex=True)]
df.columns = [col.strip() for col in df.columns]

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

In [ ]:
print("Missing values:")
display(df.isnull().sum())

print("Summary statistics:")
display(df.describe())

print("Duplicate rows:", df.duplicated().sum())

## 2. Data Cleaning

In [ ]:
cleaned = df.copy()
for col in FEATURES + [TARGET]:
    cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

cleaned[FEATURES + [TARGET]] = cleaned[FEATURES + [TARGET]].fillna(cleaned[FEATURES + [TARGET]].median())
cleaned = cleaned.drop_duplicates()

for col in FEATURES + [TARGET]:
    cleaned = cleaned[cleaned[col] >= 0]

cleaned = cleaned.reset_index(drop=True)
print("Cleaned shape:", cleaned.shape)
display(cleaned.head())

In [ ]:
outlier_rows = []
for col in FEATURES + [TARGET]:
    q1 = cleaned[col].quantile(0.25)
    q3 = cleaned[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((cleaned[col] < lower) | (cleaned[col] > upper)).sum()
    outlier_rows.append([col, q1, q3, iqr, lower, upper, count])

outlier_summary = pd.DataFrame(outlier_rows, columns=["Feature", "Q1", "Q3", "IQR", "Lower Bound", "Upper Bound", "Outlier Count"])
display(outlier_summary)

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(cleaned[TARGET], kde=True, bins=18, color="#2563eb")
plt.title("Sales Distribution", weight="bold")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feature in zip(axes, FEATURES):
    sns.regplot(data=cleaned, x=feature, y=TARGET, ax=ax, scatter_kws={"alpha": 0.75}, line_kws={"color": "black"})
    ax.set_title(f"{feature} vs Sales", weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(cleaned[FEATURES + [TARGET]].corr(), annot=True, cmap="RdBu_r", center=0, linewidths=0.5)
plt.title("Correlation Heatmap", weight="bold")
plt.show()

In [ ]:
sns.pairplot(cleaned[FEATURES + [TARGET]], diag_kind="kde")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=cleaned[FEATURES + [TARGET]], orient="h")
plt.title("Boxplot Outlier Check", weight="bold")
plt.show()

## 4. Model Results

Run `python train.py` from the project root to train all models, persist the best model, and generate final visual assets.

In [ ]:
performance_path = PROJECT_ROOT / "models" / "model_performance.csv"
importance_path = PROJECT_ROOT / "models" / "feature_importance.csv"

if performance_path.exists():
    display(pd.read_csv(performance_path))
else:
    print("Run python train.py to create model performance results.")

if importance_path.exists():
    display(pd.read_csv(importance_path))
else:
    print("Run python train.py to create feature importance results.")